In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct, avg, round, desc


In [ ]:
spark = SparkSession.builder     .appName("StateOfData-SilverToGold")     .getOrCreate()


In [ ]:
BUCKET = "tech-challenge-state-of-data"

SILVER_PATH = f"s3://{BUCKET}/silver/state_of_data/dados_harmonizados"
GOLD_PATH = f"s3://{BUCKET}/gold"


In [ ]:
silver = spark.read.parquet(SILVER_PATH)

print(f"Total de registros Silver: {silver.count()}")


In [ ]:
silver.groupBy("periodo_pesquisa")     .count()     .orderBy("periodo_pesquisa")     .show()


## Gold - Perfil dos profissionais


In [ ]:
gold_perfil_profissionais = silver.filter(
    col("cargo_atual").isNotNull()
).groupBy(
    "periodo_pesquisa",
    "cargo_atual",
    "nivel_carreira",
    "regiao"
).agg(
    count("*").alias("quantidade_profissionais"),
    countDistinct("id_resposta").alias("profissionais_unicos")
).orderBy(
    "periodo_pesquisa",
    desc("quantidade_profissionais")
)


In [ ]:
gold_perfil_profissionais.write     .mode("overwrite")     .partitionBy("periodo_pesquisa")     .parquet(f"{GOLD_PATH}/perfil_profissionais")


## Gold - Remuneração


In [ ]:
gold_remuneracao = silver.filter(
    col("faixa_salarial").isNotNull()
).groupBy(
    "periodo_pesquisa",
    "faixa_salarial",
    "nivel_carreira"
).agg(
    count("*").alias("quantidade_profissionais"),
    countDistinct("id_resposta").alias("profissionais_unicos")
).orderBy(
    "periodo_pesquisa",
    "nivel_carreira",
    "faixa_salarial"
)


In [ ]:
gold_remuneracao.write     .mode("overwrite")     .partitionBy("periodo_pesquisa")     .parquet(f"{GOLD_PATH}/remuneracao")


## Gold - Mercado de trabalho


In [ ]:
gold_mercado_trabalho = silver.filter(
    col("oportunidade_buscada").isNotNull()
).groupBy(
    "periodo_pesquisa",
    "oportunidade_buscada",
    "regiao",
    "nivel_carreira"
).agg(
    count("*").alias("quantidade_profissionais")
).orderBy(
    "periodo_pesquisa",
    desc("quantidade_profissionais")
)


In [ ]:
gold_mercado_trabalho.write     .mode("overwrite")     .partitionBy("periodo_pesquisa")     .parquet(f"{GOLD_PATH}/mercado_trabalho")


## Gold - Tecnologias


In [ ]:
gold_tecnologias = silver.filter(
    col("cloud_dia_a_dia").isNotNull()
).groupBy(
    "periodo_pesquisa",
    "cloud_dia_a_dia",
    "cloud_preferida"
).agg(
    count("*").alias("quantidade_profissionais")
).orderBy(
    "periodo_pesquisa",
    desc("quantidade_profissionais")
)


In [ ]:
gold_tecnologias.write     .mode("overwrite")     .partitionBy("periodo_pesquisa")     .parquet(f"{GOLD_PATH}/tecnologias")


## Gold - Inteligência Artificial


In [ ]:
gold_inteligencia_artificial = silver.filter(
    col("uso_ia_trabalho").isNotNull()
).groupBy(
    "periodo_pesquisa",
    "uso_ia_trabalho",
    "nivel_carreira",
    "regiao"
).agg(
    count("*").alias("quantidade_profissionais")
).orderBy(
    "periodo_pesquisa",
    desc("quantidade_profissionais")
)


In [ ]:
gold_inteligencia_artificial.write     .mode("overwrite")     .partitionBy("periodo_pesquisa")     .parquet(f"{GOLD_PATH}/inteligencia_artificial")


## Gold - Diversidade


In [ ]:
gold_diversidade = silver.filter(
    col("genero").isNotNull()
).groupBy(
    "periodo_pesquisa",
    "genero",
    "regiao",
    "nivel_carreira"
).agg(
    count("*").alias("quantidade_profissionais"),
    countDistinct("id_resposta").alias("profissionais_unicos")
).orderBy(
    "periodo_pesquisa",
    desc("quantidade_profissionais")
)


In [ ]:
gold_diversidade.write     .mode("overwrite")     .partitionBy("periodo_pesquisa")     .parquet(f"{GOLD_PATH}/diversidade")


## Validação das tabelas Gold


In [ ]:
tabelas_gold = {
    "perfil_profissionais": f"{GOLD_PATH}/perfil_profissionais",
    "remuneracao": f"{GOLD_PATH}/remuneracao",
    "mercado_trabalho": f"{GOLD_PATH}/mercado_trabalho",
    "tecnologias": f"{GOLD_PATH}/tecnologias",
    "inteligencia_artificial": f"{GOLD_PATH}/inteligencia_artificial",
    "diversidade": f"{GOLD_PATH}/diversidade"
}

for nome, caminho in tabelas_gold.items():
    df = spark.read.parquet(caminho)
    print(f"{nome}: {df.count()} registros")
